# Multi-CW Phase-Connected Sampler — Annealing + Adaptive Covariance

This notebook samples multiple continuous gravitational wave (CW) sources
in a pulsar timing array, using a simplified white-noise-only likelihood.

## Sampler Architecture

The sampler uses **three phases**:

1. **Annealing** (T_start → 1): Simulated annealing with Fisher eigenmode proposals.
   The temperature starts high (T=5000) so the chain can explore broadly, then
   geometrically cools to T=1. Proposals are scaled by √T so they're wide during
   exploration and narrow as we approach the posterior. Per-eigenmode scale adaptation
   targets ~35% acceptance.

2. **Adaptive covariance** (T=1): Uses the empirical covariance of the chain from
   the second half of annealing to build better-adapted proposals. This replaces
   the Fisher eigenmodes with data-driven directions that capture the actual
   posterior correlations.

3. **Production** (T=1): Continued sampling with the adapted proposals. This is
   the chain used for inference.

## Proposal Types

- **Eigenmode proposals** (50%): Propose along one eigenvector of the Fisher/empirical
  covariance matrix. Good for moving along narrow degeneracy directions.
- **Distance proposals** (30%): Draw a new pulsar distance from the EM prior, scan
  a grid around it to find the best distance fringe, accept via MH.
- **Joint CW proposals** (20%): Full-dimensional Gaussian proposal using the
  Cholesky of the covariance matrix. Enables correlated moves across all CW params.

## Key Features
- **Newton distance snapping** after CW parameter updates
- **Simulated annealing** for global exploration followed by adaptive local sampling
- Configurable `N_CW`, `Npulsars`, and `log10_h` at the top

## Configuration and Imports

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

# =============================================================================
# CONFIGURABLE PARAMETERS — CHANGE THESE
# =============================================================================
Npulsars = 5        # Number of pulsars to include (max 116 available)
N_CW = 3            # Number of CW sources to inject and recover
log10_h = -13.0     # Common log10(strain amplitude). -12 = high SNR, -13 = moderate

# Sampler schedule: 3 phases
n_anneal = 15000    # Phase 1: annealing steps (T: T_start -> 1)
n_adapt  = 5000     # Phase 2: adaptive covariance burn-in at T=1
n_prod   = 5000     # Phase 3: production sampling at T=1
T_start  = 5000.0   # Starting temperature (higher = broader initial exploration)
T_end    = 1.0      # Final temperature (1.0 = sample the true posterior)

# Derived constants
NCW8 = 8 * N_CW              # Total CW parameters (8 per source)
Ndim = NCW8 + Npulsars       # Total dimensions (CW params + pulsar distances)
sigma_toa = 1e-6             # White noise level [seconds] applied to all pulsars
KPC_OVER_C = disco_const.kpc / disco_const.c  # kpc -> light-seconds conversion

print(f"Configuration: N_CW={N_CW}, Npulsars={Npulsars}, h=10^{log10_h}")
print(f"Dimensions: {NCW8} CW params + {Npulsars} distances = {Ndim} total")
print(f"Sampler: {n_anneal} anneal + {n_adapt} adapt + {n_prod} prod = {n_anneal+n_adapt+n_prod} total")

## Load Pulsars and EM Distance Priors

Each pulsar has an electromagnetic (EM) distance measurement with uncertainty,
stored in `pdist = [mean, sigma]`. These become Gaussian priors on the pulsar
distance parameters in the sampler.

In [ ]:
feather_dir = "../data_products/"
disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]

# Override TOA errors with uniform white noise level
for psr in disco_psrs:
    psr.toaerrs = np.full_like(psr.toas, 1e-6, dtype=np.float32)

# Load EM distance priors from enterprise pulsar objects
psrs_ent = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}
dist_mu, dist_sig = [], []
for psr in disco_psrs:
    ep = ent_by_name[psr.name]
    mu = float(ep.pdist[0])                                # EM distance mean [kpc]
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5 # EM distance sigma [kpc]
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5  # fallback if missing
    dist_mu.append(mu)
    dist_sig.append(sig)

# JAX arrays for use in logp
dist_mu_jnp = jnp.array(dist_mu, dtype=jnp.float64)
dist_sig_jnp = jnp.array(dist_sig, dtype=jnp.float64)
sd_arr = np.array(dist_sig)   # numpy versions for proposals
mu_arr = np.array(dist_mu)
sd_jnp = jnp.array(sd_arr, dtype=jnp.float64)

# Pre-extract TOAs and sky positions for each pulsar
psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list  = [psr.pos for psr in disco_psrs]  # unit 3-vectors
psr_positions = jnp.array([psr.pos for psr in disco_psrs])  # (Npulsars, 3) for vmap

print(f"Loaded {Npulsars} pulsars: {[p.name for p in disco_psrs]}")

## Inject CW Sources and Build Truth Vector

The first source uses fixed parameters; additional sources are randomly generated.

### CW Parameter Layout (8 per source)
| Index | Name | Range | Description |
|-------|------|-------|-------------|
| 0 | cos_gwtheta | [-1, 1] | Cosine of GW source declination |
| 1 | gwphi | [0, 2pi] | GW source right ascension |
| 2 | cos_inc | [-1, 1] | Cosine of binary orbital inclination |
| 3 | log10_mc | [7, 10] | Log10 chirp mass [Msun] |
| 4 | log10_fgw | [-9, -7] | Log10 GW frequency [Hz] |
| 5 | log10_h | [-18, -11] | Log10 strain amplitude |
| 6 | phase0 | [0, 2pi] | GW phase at reference epoch |
| 7 | psi | [0, pi] | GW polarisation angle |

In [ ]:
cw_func = make_phase_connected_binary(pulsarterm=True)
CW_PARAM_NAMES = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc',
                   'log10_fgw', 'log10_h', 'phase0', 'psi']

# ----- Generate injection parameters -----
rng_inj = np.random.default_rng(12345)
INJ_LIST = [
    # Source 0: fixed parameters for reproducibility
    {"cos_gwtheta": 0.3, "gwphi": 2.5, "cos_inc": -0.2,
     "log10_mc": 9.0, "log10_fgw": -8.0, "log10_h": log10_h,
     "phase0": 1.0, "psi": 0.7},
]
# Additional sources: random parameters
for s in range(1, N_CW):
    INJ_LIST.append({
        "cos_gwtheta": float(rng_inj.uniform(-1, 1)),
        "gwphi":       float(rng_inj.uniform(0, 2*np.pi)),
        "cos_inc":     float(rng_inj.uniform(-1, 1)),
        "log10_mc":    float(rng_inj.uniform(8.5, 9.5)),
        "log10_fgw":   float(rng_inj.uniform(-8.5, -7.5)),
        "log10_h":     log10_h,
        "phase0":      float(rng_inj.uniform(0, 2*np.pi)),
        "psi":         float(rng_inj.uniform(0, np.pi)),
    })

# ----- True pulsar distances (offset from EM prior mean) -----
# We offset by 0.3 sigma so the sampler has to actually find the distances
DIST_OFFSET_SIGMA = 0.3
true_dists = {psr.name: float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])
              for i, psr in enumerate(disco_psrs)}

# ----- Compute per-source SNR -----
print("=== Injected Sources ===")
for s, inj in enumerate(INJ_LIST):
    snr2 = sum(
        np.sum(np.array(cw_func(np.asarray(psr.toas, dtype=np.float64),
               psr.pos, p_dist=true_dists[psr.name], **inj))**2) / sigma_toa**2
        for psr in disco_psrs
    )
    print(f"  src{s}: SNR={np.sqrt(snr2):.1f}, fgw=10^{inj['log10_fgw']:.2f}")

# ----- Generate data: sum of all CW signals in each pulsar -----
data_list = []
for i, psr in enumerate(disco_psrs):
    toas_i = np.asarray(psr.toas, dtype=np.float64)
    total_delay = np.zeros_like(toas_i)
    for inj in INJ_LIST:
        total_delay += np.array(cw_func(toas_i, psr.pos, p_dist=true_dists[psr.name], **inj))
    data_list.append(total_delay)

# ----- Build truth vector (no frequency sorting, matches working annealing_v2) -----
truth_cw = []
for inj in INJ_LIST:
    truth_cw += [inj[k] for k in CW_PARAM_NAMES]
truth_dist = [true_dists[psr.name] for psr in disco_psrs]
truth = np.array(truth_cw + truth_dist)

data_jnp = [jnp.array(d) for d in data_list]

# ----- Parameter bounds -----
CW_BOUNDS_LO = jnp.array([-1.0, 0.0,       -1.0, 7.0, -9.0, -18.0, 0.0,       0.0])
CW_BOUNDS_HI = jnp.array([ 1.0, 2*jnp.pi,  1.0, 10.0, -7.0, -11.0, 2*jnp.pi, jnp.pi])
PARAM_LO = np.array([float(CW_BOUNDS_LO[i%8]) for i in range(NCW8)] + [1e-6]*Npulsars)
PARAM_HI = np.array([float(CW_BOUNDS_HI[i%8]) for i in range(NCW8)] + [30.0]*Npulsars)

## Log-Posterior and Helper Functions

The log-posterior has three components:
1. **Bounds check**: All CW params within prior bounds, distances > 0
2. **Gaussian likelihood**: -1/2 sum (data - model)^2 / sigma^2 summed over all pulsars
3. **Gaussian distance prior**: -1/2 sum ((d - mu) / sigma)^2 from EM measurements

In [ ]:
@jax.jit
def logp(x):
    """Log-posterior for N_CW continuous wave sources + Npulsar distances.
    
    Parameter vector layout:
      x[0:8]        = source 0 CW params
      x[8:16]       = source 1 CW params
      ...
      x[NCW8:NCW8+Npulsars] = pulsar distances [kpc]
    """
    p_dists = x[NCW8:NCW8 + Npulsars]
    
    # --- Bounds check ---
    in_bounds = jnp.all(p_dists > 1e-6)
    for s in range(N_CW):
        cw_s = x[s*8:(s+1)*8]
        in_bounds = in_bounds & jnp.all(cw_s >= CW_BOUNDS_LO) & jnp.all(cw_s <= CW_BOUNDS_HI)
    
    # --- Gaussian likelihood ---
    # For each pulsar: residual = data - sum_of_CW_models, then chi-squared
    ll = 0.0
    for p_idx in range(Npulsars):
        model = jnp.zeros_like(data_jnp[p_idx])
        for s in range(N_CW):
            off = s * 8
            model = model + cw_func(
                psr_toas_list[p_idx], psr_pos_list[p_idx],
                cos_gwtheta=x[off+0], gwphi=x[off+1], cos_inc=x[off+2],
                log10_mc=x[off+3], log10_fgw=x[off+4], log10_h=x[off+5],
                phase0=x[off+6], psi=x[off+7],
                p_dist=p_dists[p_idx], p_phase=None)
        resid = data_jnp[p_idx] - model
        ll -= 0.5 * jnp.sum(resid**2) / sigma_toa**2
    
    # --- Gaussian distance prior ---
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu_jnp) / sd_jnp))
    
    return jnp.where(in_bounds, ll + log_prior, -1e30)


# Gradient of logp (used for Newton distance snapping)
grad_logp = jax.jit(jax.grad(logp))

# Batched logp for distance grid scans
batch_logp = jax.jit(jax.vmap(logp))


@jax.jit
def compute_delta_L_single(cos_gwtheta, gwphi, log10_fgw):
    """Compute the distance fringe spacing delta_L for each pulsar.
    
    The pulsar term creates periodic fringes in the likelihood as a
    function of pulsar distance. The fringe spacing is:
        delta_L = 1 / (f_gw * (d_kpc/c) * |1 - cos mu|)
    where mu is the angle between the GW source and the pulsar.
    Proposals need to respect this scale to avoid getting stuck.
    """
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(lambda pos: fpcmu_fast(pos, gwtheta, gwphi))(psr_positions)
    denom = jnp.maximum(jnp.abs(1.0 - cos_mu), 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)


def compute_min_delta_L(x):
    """Minimum fringe spacing across all sources, for each pulsar."""
    all_dL = []
    for s in range(N_CW):
        off = s * 8
        dL_s = np.array(compute_delta_L_single(x[off+0], x[off+1], x[off+4]))
        all_dL.append(dL_s)
    return np.min(all_dL, axis=0)


# Compile and test
print("Compiling logp...")
t0 = time.time()
lp_truth = float(logp(jnp.array(truth)))
print(f"logp(truth) = {lp_truth:.4f} (compiled in {time.time()-t0:.1f}s)")

## Fisher Information (Hessian at Truth)

We compute the Hessian of log-posterior at the true parameters to get the
Fisher information matrix. This gives us:
- **Eigenmode directions** of the posterior (principal axes of the ellipsoid)
- **Eigenmode widths** (proposal step sizes along each direction)
- **Distance curvatures** (for Newton snapping)

The Fisher covariance is `(-H)^{-1}` where H is the Hessian. We eigendecompose
this to get orthogonal proposal directions.

In [ ]:
print("Computing Hessian at truth...")
t0 = time.time()
H_full = np.array(jax.hessian(logp)(jnp.array(truth)))
print(f"Hessian computed in {time.time()-t0:.1f}s")

# ----- CW block: eigendecompose to get Fisher proposal directions -----
H_cw = H_full[:NCW8, :NCW8]                          # CW-CW block of Hessian
eig_cw, evec_cw = np.linalg.eigh(-H_cw)              # eigenvalues of Fisher info
eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())  # clip tiny eigenvalues

# Fisher covariance = inverse of Fisher information
cov_fisher = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
cov_fisher = 0.5 * (cov_fisher + cov_fisher.T)  # enforce symmetry

# Eigendecompose the covariance to get proposal directions and widths
eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_fisher)
eig_sigs_fisher = 2.38 * np.sqrt(np.maximum(eig_vals_cov, 1e-30))  # optimal MH scale

# ----- Distance block: diagonal curvatures for Newton snapping -----
H_dist_diag = np.array([H_full[NCW8+j, NCW8+j] for j in range(Npulsars)])

# Per-parameter Fisher sigma (for building the starting point)
fisher_sig_cw = np.sqrt(np.diag(cov_fisher))

# Distance fringe spacing at truth
dL = compute_min_delta_L(truth)

print(f"Fisher eigenmode widths: min={eig_sigs_fisher.min():.2e}, max={eig_sigs_fisher.max():.2e}")

# Pre-compile batch_logp
print("Pre-compiling batch_logp...")
_ = batch_logp(jnp.tile(jnp.array(truth), (40, 1)))
print("Done.")

## Sampler Helper Functions

In [ ]:
def snap_distances(x_prop, n_newton=3):
    """Newton-snap all pulsar distances toward their local likelihood peak.
    
    After updating CW parameters, the optimal distance shifts. Rather than
    waiting for the MCMC to slowly find it, we do a few Newton steps using
    the gradient and diagonal Hessian curvature:
        d_new = d_old - gradient / curvature
    This dramatically improves mixing for distance parameters.
    """
    for _ in range(n_newton):
        g = np.array(grad_logp(jnp.array(x_prop)))
        for j in range(Npulsars):
            if H_dist_diag[j] < -1e-6:  # only snap if curvature is well-defined
                x_prop[NCW8+j] = max(x_prop[NCW8+j] - g[NCW8+j] / H_dist_diag[j], 1e-6)
    return x_prop


def in_bounds(x_prop):
    """Check if all parameters are within prior bounds."""
    if np.any(x_prop[NCW8:] <= 1e-6):
        return False
    for s in range(N_CW):
        cw = x_prop[s*8:(s+1)*8]
        if np.any(cw < PARAM_LO[s*8:(s+1)*8]) or np.any(cw > PARAM_HI[s*8:(s+1)*8]):
            return False
    return True


print("Helper functions defined.")

## Starting Point (>3 sigma from truth)

We deliberately start the chain far from truth (3-5 Fisher sigmas away in each
parameter) to test that the annealing can actually find the posterior mode.

In [ ]:
rng = np.random.default_rng(99)
x0 = truth.copy()

# Offset CW parameters by 3-5 Fisher sigmas in a random direction
for i in range(NCW8):
    offset_sigma = rng.uniform(3.0, 5.0) * rng.choice([-1, 1])
    proposed = x0[i] + offset_sigma * fisher_sig_cw[i]
    lo = float(CW_BOUNDS_LO[i % 8]) + 1e-4
    hi = float(CW_BOUNDS_HI[i % 8]) - 1e-4
    x0[i] = np.clip(proposed, lo, hi)

# Offset distances by 3-5 EM sigmas
for j in range(Npulsars):
    offset = rng.uniform(3.0, 5.0) * rng.choice([-1, 1]) * sd_arr[j]
    x0[NCW8+j] = max(truth[NCW8+j] + offset, 0.01)

lp_start = float(logp(jnp.array(x0)))
print(f"logp(start) = {lp_start:.2f}")
print(f"logp(truth) = {lp_truth:.4f}")
print(f"Gap: {lp_truth - lp_start:.0f} nats")

## Run the Three-Phase Sampler

### Phase 1: Annealing (steps 0 to n_anneal)
Temperature cools geometrically: `T(step) = T_start * r^step` where `r = (1/T_start)^(1/n_anneal)`.
Proposals are scaled by sqrt(T) so they're broad when T is high.
Samples from the second half of annealing are saved to build the empirical covariance.

### Phase 2: Adaptive covariance (steps n_anneal to n_anneal + n_adapt)
At the phase transition, we compute the empirical covariance from annealing samples
and switch proposals to use it. This adapts proposal directions to the actual
posterior shape (which may differ from the Fisher approximation at truth).

### Phase 3: Production (steps n_anneal + n_adapt to end)
Standard MCMC at T=1 with the adapted proposals. This is the chain used for inference.

In [ ]:
# Geometric cooling rate: T(step) = T_start * cool_rate^step
cool_rate = (T_end / T_start) ** (1.0 / n_anneal)
n_total = n_anneal + n_adapt + n_prod

print(f"Annealing: T {T_start} -> {T_end} over {n_anneal} steps (cool_rate={cool_rate:.6f})")
print(f"Adapt:     {n_adapt} steps at T=1 to build empirical covariance")
print(f"Prod:      {n_prod} steps")

# ----- Storage for the full chain -----
all_chain = np.zeros((n_total, Ndim))   # parameter values at each step
all_lps   = np.zeros(n_total)           # log-posterior at each step
all_temps = np.zeros(n_total)           # temperature at each step

# ----- Sampler state -----
x  = x0.copy()                          # current position
lp = float(logp(jnp.array(x)))          # current log-posterior
T  = T_start                            # current temperature

# ----- Empirical covariance accumulator -----
# During annealing (2nd half), we save samples to build empirical covariance
emp_samples = []     # list of parameter vectors
emp_cov = None       # will hold {'L': Cholesky, 'eig_sigs': widths, 'vecs': eigenvectors}
use_emp_cov = False  # switch to True after building empirical cov

# ----- Per-eigenmode scale adaptation (annealing only) -----
# Each eigenmode has a log-scale factor that's adapted to target 35% acceptance
scale_log = np.zeros(NCW8)

# ----- Acceptance tracking -----
acc_counts = {'eigen': 0, 'dist': 0, 'joint': 0}
tot_counts = {'eigen': 0, 'dist': 0, 'joint': 0}

t0 = time.time()

for step in range(n_total):
    # ----- Determine current phase and temperature -----
    if step < n_anneal:
        phase = 'anneal'
        T = T_start * (cool_rate ** step)  # geometric cooling
    else:
        phase = 'adapt' if step < n_anneal + n_adapt else 'prod'
        T = 1.0
    
    # ----- Phase transition: build empirical covariance -----
    if step == n_anneal and len(emp_samples) > NCW8 * 2:
        emp_arr = np.array(emp_samples)
        # Use last ~1/3 of annealing samples (when T was closest to 1)
        last_n = max(NCW8 * 3, len(emp_arr) // 3)
        emp_arr = emp_arr[-last_n:, :NCW8]  # CW params only
        
        if emp_arr.shape[0] > NCW8:
            raw_cov = np.cov(emp_arr.T)
            raw_cov = 0.5 * (raw_cov + raw_cov.T)  # enforce symmetry
            
            # Regularise: clip tiny eigenvalues to avoid singular covariance
            eig_e, vec_e = np.linalg.eigh(raw_cov)
            eig_e = np.maximum(eig_e, 1e-12 * eig_e.max())
            emp_cov_matrix = vec_e @ np.diag(eig_e) @ vec_e.T
            
            # Scale by 2.38^2/d (optimal MH scaling for Gaussians)
            scale_nd = 2.38**2 / NCW8
            try:
                L_emp = np.linalg.cholesky(scale_nd * emp_cov_matrix)
                emp_cov = {
                    'L': L_emp,                              # Cholesky factor for joint proposals
                    'eig_sigs': 2.38 * np.sqrt(eig_e),      # eigenmode widths
                    'vecs': vec_e,                           # eigenvectors
                }
                use_emp_cov = True
                print(f"  [step {step}] Switched to empirical covariance "
                      f"(built from {emp_arr.shape[0]} samples)")
            except np.linalg.LinAlgError:
                print(f"  [step {step}] Empirical Cholesky failed, keeping Fisher cov")
    
    # ----- Choose and execute a proposal -----
    r = rng.random()
    
    if r < 0.50:
        # === EIGENMODE PROPOSAL (50% of steps) ===
        # Pick one eigenvector of the covariance, propose a Gaussian step along it.
        # During annealing: use Fisher eigenmodes scaled by sqrt(T)
        # After transition: use empirical eigenmodes
        if use_emp_cov:
            mode_idx = rng.integers(NCW8)
            z = rng.standard_normal()
            sig = emp_cov['eig_sigs'][mode_idx]
            x_prop = x.copy()
            x_prop[:NCW8] += z * sig * emp_cov['vecs'][:, mode_idx]
        else:
            mode_idx = rng.integers(NCW8)
            z = rng.standard_normal()
            # Scale by sqrt(T) (wider when hot) and per-mode adaptation factor
            scaled_sig = eig_sigs_fisher[mode_idx] * np.sqrt(T) * np.exp(scale_log[mode_idx])
            x_prop = x.copy()
            x_prop[:NCW8] += z * scaled_sig * eig_vecs_cov[:, mode_idx]
        
        # Newton-snap distances to their local optimum after CW param change
        x_prop = snap_distances(x_prop)
        
        if in_bounds(x_prop):
            lp_prop = float(logp(jnp.array(x_prop)))
            # Tempered Metropolis-Hastings acceptance: log alpha = (logp_prop - logp) / T
            log_alpha = (lp_prop - lp) / T
            accepted = np.log(rng.random() + 1e-300) < log_alpha
            if accepted:
                x = x_prop; lp = lp_prop
            acc_counts['eigen'] += int(accepted)
        tot_counts['eigen'] += 1
        
        # Per-eigenmode scale adaptation (annealing only)
        # Robbins-Monro: nudge scale toward 35% acceptance
        if phase == 'anneal':
            gamma = 1.0 / (step + 100)  # decaying step size
            scale_log[mode_idx] += gamma * (float(x is x_prop) - 0.35)
            scale_log[mode_idx] = np.clip(scale_log[mode_idx], -5.0, 10.0)
    
    elif r < 0.80:
        # === DISTANCE PROPOSAL (30% of steps) ===
        # Draw a new distance from the EM prior, then scan a grid around it
        # to find the best distance fringe. This handles the highly multimodal
        # distance likelihood (many narrow fringes from the pulsar term).
        pi = rng.integers(Npulsars)
        d_prop = rng.normal(mu_arr[pi], sd_arr[pi] * max(1.0, np.sqrt(T)))
        dL_j = float(dL[pi])  # fringe spacing for this pulsar
        
        if d_prop > dL_j:  # must be positive and > one fringe width
            x_snap = x.copy()
            x_snap[NCW8+pi] = d_prop
            
            # Scan a grid of +/-0.6 fringe widths to find the best fringe
            d_lo = max(d_prop - 0.6 * dL_j, 1e-6)
            d_hi = d_prop + 0.6 * dL_j
            d_cands = np.linspace(d_lo, d_hi, 30)
            x_batch = np.tile(x_snap, (30, 1))
            x_batch[:, NCW8+pi] = d_cands
            lps_scan = np.array(batch_logp(jnp.array(x_batch)))
            
            # Pick the best candidate from the grid
            x_prop = x.copy()
            x_prop[NCW8+pi] = float(d_cands[np.argmax(lps_scan)])
            lp_prop = float(logp(jnp.array(x_prop)))
            
            # Tempered MH acceptance
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['dist'] += 1
        tot_counts['dist'] += 1
    
    else:
        # === JOINT CW PROPOSAL (20% of steps) ===
        # Full-dimensional Gaussian proposal using the Cholesky factor of
        # the covariance matrix. Enables correlated moves across all CW params.
        if use_emp_cov:
            z = rng.standard_normal(NCW8)
            x_prop = x.copy()
            x_prop[:NCW8] += emp_cov['L'] @ z
        else:
            scale_nd = 2.38**2 / NCW8
            L_fish = np.linalg.cholesky(scale_nd * T * cov_fisher)
            z = rng.standard_normal(NCW8)
            x_prop = x.copy()
            x_prop[:NCW8] += L_fish @ z
        
        if in_bounds(x_prop):
            lp_prop = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['joint'] += 1
        tot_counts['joint'] += 1
    
    # ----- Record state -----
    all_chain[step] = x
    all_lps[step]   = lp
    all_temps[step]  = T
    
    # Accumulate samples for empirical covariance (last half of annealing)
    if phase == 'anneal' and step > n_anneal // 2:
        emp_samples.append(x.copy())
    
    # Progress logging
    if step % 2000 == 0:
        elapsed = time.time() - t0
        print(f"  step {step:5d}/{n_total} [{phase:6s}] T={T:7.2f} | logp={lp:10.2f} | "
              f"eigen={acc_counts['eigen']}/{tot_counts['eigen']} "
              f"dist={acc_counts['dist']}/{tot_counts['dist']} "
              f"joint={acc_counts['joint']}/{tot_counts['joint']} | {elapsed:.0f}s")

# ----- Summary -----
dt = time.time() - t0
print(f"\nDone in {dt:.1f}s ({n_total/dt:.0f} it/s)")
for k in acc_counts:
    ar = acc_counts[k] / max(tot_counts[k], 1)
    print(f"  {k:10s}: {acc_counts[k]:5d}/{tot_counts[k]:5d} = {ar:.3f}")

## Diagnostics and Results

In [ ]:
prod_chain = all_chain[n_anneal + n_adapt:]
prod_lps   = all_lps[n_anneal + n_adapt:]

print(f"logp truth = {lp_truth:.4f}")
print(f"logp prod:  mean={np.mean(prod_lps):.2f}, std={np.std(prod_lps):.2f}, "
      f"max={np.max(prod_lps):.2f}")

print("\nCW parameter recovery (production chain):")
for s in range(N_CW):
    print(f"  --- src{s} ---")
    for pidx, pname in enumerate(CW_PARAM_NAMES):
        col = s*8 + pidx
        med = np.median(prod_chain[:, col])
        std = np.std(prod_chain[:, col])
        bias = med - truth[col]
        print(f"    {pname:15s}: truth={truth[col]:+.4f}, median={med:+.4f}, "
              f"std={std:.2e}, bias={bias:+.4f}")

print("\nDistance recovery (production chain):")
for j in range(Npulsars):
    col = NCW8 + j
    med = np.median(prod_chain[:, col])
    err_modes = abs(med - truth[col]) / dL[j]
    print(f"  {disco_psrs[j].name}: med={med:.4f}, truth={truth[col]:.4f}, "
          f"err={err_modes:.0f} fringe modes")

## Trace Plots

4x3 grid showing:
- Row 0: logp trace, temperature schedule, log10_fgw traces
- Row 1: cos_gwtheta, log10_h, cos_inc traces
- Row 2-3: pulsar distance traces + log10_h posterior histogram

In [ ]:
ann_end  = n_anneal
adap_end = n_anneal + n_adapt
steps_arr = np.arange(n_total)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
fig.suptitle(
    f'N_CW={N_CW}, h=10^{log10_h}: Annealing (T:{T_start:.0f}->1) + Adaptive Cov + Production\n'
    f'Start: dlogp={lp_start-lp_truth:.0f} from truth | '
    f'{n_anneal}+{n_adapt}+{n_prod} steps',
    fontsize=13)

colours_src = ['#1a3a5c', '#b5442d', '#2d7a3d']
src_labels = [f'src{s}' for s in range(N_CW)]

# --- Row 0, Col 0: logp trace ---
ax = axes[0, 0]
ax.plot(steps_arr, all_lps, color='#333', lw=0.3, alpha=0.8)
ax.axhline(lp_truth, color='r', ls='--', lw=1, label=f'truth ({lp_truth:.2f})')
ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.7, label='anneal end')
ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.7, label='adapt end')
ax.set_xlabel('step'); ax.set_ylabel('logp')
ax.set_title('Log-posterior trace'); ax.legend(fontsize=7)

# --- Row 0, Col 1: temperature ---
ax = axes[0, 1]
ax.semilogy(steps_arr[:n_anneal], all_temps[:n_anneal], color='#b5442d', lw=0.6)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_xlabel('step'); ax.set_ylabel('Temperature T')
ax.set_title('Temperature schedule (annealing phase)')

# --- Row 0, Col 2: log10_fgw for all sources ---
ax = axes[0, 2]
for s in range(min(N_CW, 3)):
    col = s*8 + 4
    ax.plot(steps_arr, all_chain[:, col], color=colours_src[s], lw=0.3, alpha=0.7, label=src_labels[s])
    ax.axhline(truth[col], color=colours_src[s], ls='--', lw=1)
ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.5)
ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.5)
ax.set_title('log10_fgw'); ax.set_xlabel('step'); ax.legend(fontsize=7)

# --- Row 1: cos_gwtheta, log10_h, cos_inc ---
params_show = [(0, 'cos_gwtheta'), (5, 'log10_h'), (2, 'cos_inc')]
for pidx, (param_off, param_name) in enumerate(params_show):
    ax = axes[1, pidx]
    for s in range(min(N_CW, 3)):
        col = s*8 + param_off
        ax.plot(steps_arr, all_chain[:, col], color=colours_src[s], lw=0.3, alpha=0.7, label=src_labels[s])
        ax.axhline(truth[col], color=colours_src[s], ls='--', lw=1)
        ax.plot(0, x0[col], 'o', color=colours_src[s], ms=5, zorder=5)
    ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.5)
    ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.5)
    ax.set_title(param_name); ax.set_xlabel('step'); ax.legend(fontsize=7)

# --- Rows 2-3: distance traces ---
dist_colours = ['#1a3a5c', '#8b2500', '#2d5a27', '#6a3d9a', '#b15928']
for j in range(Npulsars):
    ax = axes[2, j] if j < 3 else axes[3, j - 3]
    ax.plot(steps_arr, all_chain[:, NCW8+j], color=dist_colours[j % 5], lw=0.3, alpha=0.8)
    ax.axhline(truth[NCW8+j], color='r', ls='--', lw=1, label='truth')
    ax.plot(0, x0[NCW8+j], 'o', color='orange', ms=6, zorder=5, label='start')
    ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.5)
    ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.5)
    ax.set_title(f'{disco_psrs[j].name} dist'); ax.set_xlabel('step')
    ax.legend(fontsize=7)

# --- Row 3, Col 2: log10_h posteriors (production only) ---
ax = axes[3, 2]
for s in range(min(N_CW, 3)):
    col = s*8 + 5
    ax.hist(prod_chain[:, col], bins=50, density=True, alpha=0.5,
            color=colours_src[s], label=f'src{s} (truth={truth[col]:.1f})')
    ax.axvline(truth[col], color=colours_src[s], ls='--', lw=1.5)
ax.set_title('log10_h posteriors (production)'); ax.set_xlabel('log10_h')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()